In [26]:
"""
Calculate Vertical Hydraulic Gradient (VHG) for sites with paired well-piezometer data.

VHG = (head_well - head_piezo) / vertical_separation

Positive VHG = downward flow (recharge)
Negative VHG = upward flow (discharge)

Only processes sites that have BOTH well and piezometer measurements.
"""

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [27]:
def identify_paired_sites(data_dir, sensor_depths_file='levellogger_below_ground.csv'):
    """
    Identify which sites have both well and piezometer measurements.
    
    Parameters:
    -----------
    data_dir : Path
        Directory containing processed water level files
    sensor_depths_file : str
        Name of the sensor depth reference file
        
    Returns:
    --------
    dict
        Dictionary with site_base as key, dict with 'well' and 'piezo' info as value
    """
    data_dir = Path(data_dir)
    
    # Load sensor depths to get vertical separations
    # Look for sensor depths file in parent directory (where raw data is)
    sensor_depths_path = data_dir.parent / sensor_depths_file
    if not sensor_depths_path.exists():
        # Try in the same directory
        sensor_depths_path = data_dir / sensor_depths_file
    if not sensor_depths_path.exists():
        raise FileNotFoundError(
            f"Cannot find {sensor_depths_file} in {data_dir} or {data_dir.parent}"
        )
    
    sensor_depths = pd.read_csv(sensor_depths_path)
    
    # Get all processed files
    processed_files = list(data_dir.glob('*WaterLevelBG.csv'))
    
    # Group by site base name
    sites = {}
    for filepath in processed_files:
        # Extract site base name (e.g., 'APA' from 'APA_wellWaterLevelBG.csv')
        filename = filepath.stem  # Remove .csv
        
        # Determine if it's a well or piezo
        if 'piezo' in filename.lower():
            site_base = filename.split('_piezo')[0].replace('WaterLevelBG', '')
            logger_type = 'piezo'
            logger_id = f"{site_base}_piezo"
        elif 'well' in filename.lower():
            site_base = filename.split('_well')[0].replace('WaterLevelBG', '')
            logger_type = 'well'
            logger_id = f"{site_base}_well"
        else:
            continue  # Skip files that aren't clearly well or piezo
        
        # Initialize site if not exists
        if site_base not in sites:
            sites[site_base] = {'well': None, 'piezo': None}
        
        # Get sensor depth
        depth_match = sensor_depths[sensor_depths['logger_ID'] == logger_id]
        if len(depth_match) > 0:
            sensor_depth = depth_match['sensor_below_ground'].values[0]
        else:
            sensor_depth = None
        
        # Store info
        sites[site_base][logger_type] = {
            'filepath': filepath,
            'logger_id': logger_id,
            'sensor_depth': sensor_depth
        }
    
    # Filter to only sites with BOTH well and piezo
    paired_sites = {
        site: info for site, info in sites.items() 
        if info['well'] is not None and info['piezo'] is not None
    }
    
    return paired_sites


In [28]:
def calculate_vhg_for_site(site_name, site_info, output_dir):
    """
    Calculate VHG for a single site with paired well-piezo data.
    
    Parameters:
    -----------
    site_name : str
        Base site name (e.g., 'APA', 'CCA')
    site_info : dict
        Dictionary with 'well' and 'piezo' information
    output_dir : Path
        Directory to save VHG results
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with VHG calculations
    """
    print(f"\nProcessing site: {site_name}")
    print("-" * 60)
    
    # Load well data
    well_data = pd.read_csv(site_info['well']['filepath'])
    well_data['datetime'] = pd.to_datetime(well_data['datetime'])
    well_data = well_data.set_index('datetime')
    
    # Load piezo data
    piezo_data = pd.read_csv(site_info['piezo']['filepath'])
    piezo_data['datetime'] = pd.to_datetime(piezo_data['datetime'])
    piezo_data = piezo_data.set_index('datetime')
    
    print(f"Well logger:  {site_info['well']['logger_id']}")
    print(f"  Sensor depth: {site_info['well']['sensor_depth']} cm")
    print(f"  Records: {len(well_data)}")
    print(f"  Date range: {well_data.index.min()} to {well_data.index.max()}")
    
    print(f"Piezo logger: {site_info['piezo']['logger_id']}")
    print(f"  Sensor depth: {site_info['piezo']['sensor_depth']} cm")
    print(f"  Records: {len(piezo_data)}")
    print(f"  Date range: {piezo_data.index.min()} to {piezo_data.index.max()}")
    
    # Check if we have sensor depths
    if site_info['well']['sensor_depth'] is None or site_info['piezo']['sensor_depth'] is None:
        print("  WARNING: Missing sensor depth information - cannot calculate VHG")
        return None
    
    # Merge on datetime (inner join to get matching timestamps only)
    merged = well_data[['water_level_below_ground_cm']].join(
        piezo_data[['water_level_below_ground_cm']], 
        how='inner',
        rsuffix='_piezo', 
        lsuffix='_well'
    )
    
    print(f"Matched records: {len(merged)}")
    
    if len(merged) == 0:
        print("  WARNING: No overlapping timestamps - cannot calculate VHG")
        return None
    
    # Calculate vertical separation between sensors (cm)
    well_depth = site_info['well']['sensor_depth']
    piezo_depth = site_info['piezo']['sensor_depth']
    vertical_separation = abs(well_depth - piezo_depth)
    
    print(f"Vertical separation: {vertical_separation:.1f} cm")
    
    # Calculate VHG (dimensionless)
    # VHG = (head_well - head_piezo) / vertical_separation
    # Positive = downward flow (recharge)
    # Negative = upward flow (discharge)
    merged['VHG'] = ((merged['water_level_below_ground_cm_well'] - 
                      merged['water_level_below_ground_cm_piezo']) / vertical_separation)
    
    # Add site information
    merged['site'] = site_name
    merged['well_logger'] = site_info['well']['logger_id']
    merged['piezo_logger'] = site_info['piezo']['logger_id']
    merged['vertical_separation_cm'] = vertical_separation
    
    # Calculate statistics
    print(f"\nVHG Statistics:")
    print(f"  Mean: {merged['VHG'].mean():8.6f}")
    print(f"  Median: {merged['VHG'].median():8.6f}")
    print(f"  Std: {merged['VHG'].std():8.6f}")
    print(f"  Min: {merged['VHG'].min():8.6f}")
    print(f"  Max: {merged['VHG'].max():8.6f}")
    
    # Interpret predominant flow direction
    mean_vhg = merged['VHG'].mean()
    if mean_vhg > 0.01:
        flow_dir = "DOWNWARD flow (recharge conditions)"
    elif mean_vhg < -0.01:
        flow_dir = "UPWARD flow (discharge conditions)"
    else:
        flow_dir = "MINIMAL vertical flow"
    print(f"  Interpretation: {flow_dir}")
    
    # Count flow direction occurrences
    downward_pct = (merged['VHG'] > 0).sum() / len(merged) * 100
    upward_pct = (merged['VHG'] < 0).sum() / len(merged) * 100
    print(f"  Downward flow: {downward_pct:.1f}% of time")
    print(f"  Upward flow: {upward_pct:.1f}% of time")
    
    # Save to CSV
    output_file = output_dir / f"{site_name}_VHG.csv"
    merged.to_csv(output_file)
    print(f"  Saved to: {output_file.name}")
    
    # Save to CSV
    output_file = output_dir / f"{site_name}_VHG.csv"
    merged.to_csv(output_file)
    print(f"  Saved to: {output_file.name}")
    
    # ADD THESE TWO LINES HERE:
    save_paired_water_levels_csv(site_name, merged, output_dir)
    plot_paired_water_levels(site_name, merged, output_dir)
    
    return merged

def calculate_all_vhg(data_dir, output_dir=None):
    """
    Calculate VHG for all sites with paired well-piezometer data.
    
    Parameters:
    -----------
    data_dir : str or Path
        Directory containing processed WaterLevelBG.csv files
    output_dir : str or Path, optional
        Directory to save VHG results. If None, creates 'VHG_analysis' subdirectory
        
    Returns:
    --------
    dict
        Dictionary mapping site names to VHG DataFrames
    """
    data_dir = Path(data_dir)
    
    # Set up output directory
    if output_dir is None:
        output_dir = data_dir / 'VHG_analysis'
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    print("=" * 80)
    print("VERTICAL HYDRAULIC GRADIENT (VHG) CALCULATION")
    print("=" * 80)
    print(f"Data directory: {data_dir}")
    print(f"Output directory: {output_dir}")
    
    # Identify paired sites
    print("\nIdentifying sites with paired well-piezometer data...")
    paired_sites = identify_paired_sites(data_dir)
    
    print(f"\nFound {len(paired_sites)} sites with paired data:")
    for site in sorted(paired_sites.keys()):
        print(f"  - {site}")
    
    if len(paired_sites) == 0:
        print("\nNo paired sites found! Cannot calculate VHG.")
        return {}
    
    print("\n" + "=" * 80)
    
    # Calculate VHG for each site
    vhg_results = {}
    for site_name, site_info in sorted(paired_sites.items()):
        try:
            vhg_df = calculate_vhg_for_site(site_name, site_info, output_dir)
            if vhg_df is not None:
                vhg_results[site_name] = vhg_df
        except Exception as e:
            print(f"  ERROR processing {site_name}: {e}")
            import traceback
            traceback.print_exc()
    
    print("\n" + "=" * 80)
    print(f"VHG calculation complete for {len(vhg_results)} sites!")
    
    # Create combined VHG file
    if vhg_results:
        print("\nCreating combined VHG file...")
        combined = pd.concat(vhg_results.values(), ignore_index=False)
        combined_file = output_dir / 'all_sites_VHG.csv'
        combined.to_csv(combined_file)
        print(f"Combined VHG saved to: {combined_file.name}")
        
        # Create summary statistics
        create_vhg_summary(vhg_results, output_dir)
    
    return vhg_results

In [29]:
def save_paired_water_levels_csv(site_name, vhg_df, output_dir):
    """
    Save a clean CSV with datetime, well water level, piezo water level, and VHG
    for easy inspection of the paired measurements.
    
    Parameters:
    -----------
    site_name : str
        Site name
    vhg_df : pd.DataFrame
        VHG results dataframe
    output_dir : Path
        Output directory
    """
    # Create simplified dataframe with clear column names
    paired_data = pd.DataFrame({
        'datetime': vhg_df.index,
        'well_water_level_cm': vhg_df['water_level_below_ground_cm_well'],
        'piezo_water_level_cm': vhg_df['water_level_below_ground_cm_piezo'],
        'VHG': vhg_df['VHG'],
        'vertical_separation_cm': vhg_df['vertical_separation_cm']
    })
    
    # Save to CSV
    output_file = output_dir / f"{site_name}_paired_water_levels.csv"
    paired_data.to_csv(output_file, index=False)
    print(f"  Paired water levels CSV saved to: {output_file.name}")
    
    return output_file


def plot_paired_water_levels(site_name, vhg_df, output_dir):
    """
    Create a line graph showing well and piezometer water levels over time
    for a single site.
    
    Parameters:
    -----------
    site_name : str
        Site name
    vhg_df : pd.DataFrame
        VHG results dataframe
    output_dir : Path
        Output directory
    """
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Plot water levels
    ax.plot(vhg_df.index, vhg_df['water_level_below_ground_cm_well'], 
            linewidth=1.5, color='blue', label=vhg_df['well_logger'].iloc[0], alpha=0.8)
    ax.plot(vhg_df.index, vhg_df['water_level_below_ground_cm_piezo'], 
            linewidth=1.5, color='red', label=vhg_df['piezo_logger'].iloc[0], alpha=0.8)
    
    # Invert y-axis so 0 is at top (surface) and depth increases downward
    ax.invert_yaxis()
    
    # Labels and title
    ax.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax.set_ylabel('Depth Below Ground Surface (cm)', fontsize=12, fontweight='bold')
    ax.set_title(f'{site_name} - Water Level Measurements', fontsize=14, fontweight='bold')
    
    # Add mean VHG as text
    mean_vhg = vhg_df['VHG'].mean()
    if mean_vhg > 0.01:
        flow_text = f"Mean VHG: {mean_vhg:.6f}\n(Downward flow - Recharge)"
        box_color = 'lightblue'
    elif mean_vhg < -0.01:
        flow_text = f"Mean VHG: {mean_vhg:.6f}\n(Upward flow - Discharge)"
        box_color = 'lightcoral'
    else:
        flow_text = f"Mean VHG: {mean_vhg:.6f}\n(Minimal vertical flow)"
        box_color = 'lightgray'
    
    ax.text(0.02, 0.98, flow_text, transform=ax.transAxes,
           fontsize=11, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.7))
    
    # Legend
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
    
    # Grid
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Format x-axis dates
    fig.autofmt_xdate()  # Rotate date labels
    
    plt.tight_layout()
    
    # Save
    plot_file = output_dir / f"{site_name}_paired_water_levels.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"  Paired water levels plot saved to: {plot_file.name}")
    plt.close()
    
    return plot_file

In [30]:
def create_vhg_summary(vhg_results, output_dir):
    """
    Create summary statistics table for all sites.
    """
    print("\nCreating VHG summary statistics...")
    
    summary_data = []
    for site_name, vhg_df in vhg_results.items():
        summary_data.append({
            'site': site_name,
            'n_records': len(vhg_df),
            'mean_VHG': vhg_df['VHG'].mean(),
            'median_VHG': vhg_df['VHG'].median(),
            'std_VHG': vhg_df['VHG'].std(),
            'min_VHG': vhg_df['VHG'].min(),
            'max_VHG': vhg_df['VHG'].max(),
            'pct_downward': (vhg_df['VHG'] > 0).sum() / len(vhg_df) * 100,
            'pct_upward': (vhg_df['VHG'] < 0).sum() / len(vhg_df) * 100,
            'vertical_separation_cm': vhg_df['vertical_separation_cm'].iloc[0],
            'start_date': vhg_df.index.min(),
            'end_date': vhg_df.index.max()
        })
    
    summary = pd.DataFrame(summary_data)
    summary = summary.round({
        'mean_VHG': 6,
        'median_VHG': 6,
        'std_VHG': 6,
        'min_VHG': 6,
        'max_VHG': 6,
        'pct_downward': 1,
        'pct_upward': 1,
        'vertical_separation_cm': 1
    })
    
    # Add predominant flow direction
    def flow_direction(mean_vhg):
        if mean_vhg > 0.01:
            return "Downward (recharge)"
        elif mean_vhg < -0.01:
            return "Upward (discharge)"
        else:
            return "Minimal flow"
    
    summary['predominant_flow'] = summary['mean_VHG'].apply(flow_direction)
    
    # Save
    summary_file = output_dir / 'VHG_summary_statistics.csv'
    summary.to_csv(summary_file, index=False)
    print(f"Summary statistics saved to: {summary_file.name}")
    
    # Print summary
    print("\n" + "=" * 80)
    print("VHG SUMMARY BY SITE")
    print("=" * 80)
    print(summary[['site', 'mean_VHG', 'pct_downward', 'pct_upward', 'predominant_flow']].to_string(index=False))
    
    return summary


In [31]:
def calculate_vhg_august_stats(vhg_results, output_dir):
    """
    Calculate VHG statistics for August 1 - September 1 period for each site.
    Automatically detects the year from the data.
    
    Parameters:
    -----------
    vhg_results : dict
        Dictionary mapping site names to VHG DataFrames
    output_dir : Path
        Directory to save results
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with statistics for each site
    """
    print("\n" + "=" * 80)
    print("CALCULATING VHG STATISTICS FOR AUGUST 1 - SEPTEMBER 1")
    print("=" * 80)
    
    stats_data = []
    
    for site_name, vhg_df in sorted(vhg_results.items()):
        print(f"\nProcessing {site_name}...")
        
        # Filter to August 1 - September 1 (any year in the data)
        # Use month and day filtering
        mask = ((vhg_df.index.month == 8) & (vhg_df.index.day >= 1)) | \
               ((vhg_df.index.month == 9) & (vhg_df.index.day == 1))
        august_data = vhg_df.loc[mask, 'VHG']
        
        if len(august_data) == 0:
            print(f"  No data in August 1 - September 1 period for {site_name}")
            # Still add row with NaN values
            stats_data.append({
                'SiteID': site_name,
                'VHG_average': np.nan,
                'VHG_median': np.nan,
                'VHG_max': np.nan,
                'VHG_min': np.nan,
                'VHG_stdev': np.nan,
                'VHG_range': np.nan,
                'n_records': 0
            })
        else:
            print(f"  Found {len(august_data)} records")
            print(f"  Date range: {august_data.index.min()} to {august_data.index.max()}")
            
            # Calculate statistics
            vhg_avg = august_data.mean()
            vhg_median = august_data.median()
            vhg_max = august_data.max()
            vhg_min = august_data.min()
            vhg_stdev = august_data.std()
            vhg_range = vhg_max - vhg_min
            
            print(f"  VHG average: {vhg_avg:.6f}")
            print(f"  VHG median:  {vhg_median:.6f}")
            print(f"  VHG max:     {vhg_max:.6f}")
            print(f"  VHG min:     {vhg_min:.6f}")
            print(f"  VHG stdev:   {vhg_stdev:.6f}")
            print(f"  VHG range:   {vhg_range:.6f}")
            
            stats_data.append({
                'SiteID': site_name,
                'VHG_average': vhg_avg,
                'VHG_median': vhg_median,
                'VHG_max': vhg_max,
                'VHG_min': vhg_min,
                'VHG_stdev': vhg_stdev,
                'VHG_range': vhg_range,
                'n_records': len(august_data)
            })
    
    # Create DataFrame with columns in specified order
    stats_df = pd.DataFrame(stats_data)
    stats_df = stats_df[['SiteID', 'VHG_average', 'VHG_median', 'VHG_max', 
                         'VHG_min', 'VHG_stdev', 'VHG_range', 'n_records']]
    
    # Round values
    stats_df = stats_df.round({
        'VHG_average': 6,
        'VHG_median': 6,
        'VHG_max': 6,
        'VHG_min': 6,
        'VHG_stdev': 6,
        'VHG_range': 6
    })
    
    # Save to CSV
    output_file = output_dir / 'VHG_August_stats.csv'
    stats_df.to_csv(output_file, index=False)
    
    print("\n" + "=" * 80)
    print("VHG AUGUST STATISTICS SAVED")
    print("=" * 80)
    print(f"Output file: {output_file.name}")
    print(f"Total sites: {len(stats_df)}")
    print(f"Sites with data: {(stats_df['n_records'] > 0).sum()}")
    print(f"Sites without data: {(stats_df['n_records'] == 0).sum()}")
    
    # Print the table
    print("\n" + stats_df.to_string(index=False))
    
    return stats_df

In [32]:
def create_vhg_boxplot(vhg_results, output_dir):
    """
    Create box plot comparing VHG distributions across sites.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Prepare data for box plot
    data = []
    labels = []
    for site_name, vhg_df in sorted(vhg_results.items()):
        data.append(vhg_df['VHG'].values)
        labels.append(site_name)
    
    # Create box plot
    bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    
    # Color boxes
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    
    # Add zero line
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='No vertical flow')
    
    # Formatting
    ax.set_ylabel('Vertical Hydraulic Gradient (VHG)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Site', fontsize=12, fontweight='bold')
    ax.set_title('VHG Distribution by Site', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend()
    
    # Rotate labels if needed
    if len(labels) > 5:
        plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    
    # Save
    plot_file = output_dir / 'VHG_boxplot_comparison.png'
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"VHG boxplot saved to: {plot_file.name}")
    plt.close()


In [33]:
def plot_vhg_all_sites(vhg_results, output_dir):
    """
    Create comprehensive VHG plots for all sites.
    """
    print("\nCreating VHG visualizations...")
    
    n_sites = len(vhg_results)
    if n_sites == 0:
        return
    
    # Create multi-panel plot
    fig, axes = plt.subplots(n_sites, 1, figsize=(14, 4*n_sites), sharex=False)
    
    # Handle single site case
    if n_sites == 1:
        axes = [axes]
    
    fig.suptitle('Vertical Hydraulic Gradient - All Sites', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    for idx, (site_name, vhg_df) in enumerate(sorted(vhg_results.items())):
        ax = axes[idx]
        
        # Plot VHG
        ax.plot(vhg_df.index, vhg_df['VHG'], linewidth=0.5, alpha=0.7, color='purple')
        
        # Add zero line
        ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
        
        # Fill areas
        ax.fill_between(vhg_df.index, 0, vhg_df['VHG'], 
                        where=(vhg_df['VHG'] > 0), alpha=0.3, color='blue', 
                        label='Downward flow')
        ax.fill_between(vhg_df.index, 0, vhg_df['VHG'], 
                        where=(vhg_df['VHG'] < 0), alpha=0.3, color='orange', 
                        label='Upward flow')
        
        # Formatting
        ax.set_ylabel('VHG', fontsize=11)
        ax.set_title(f'{site_name} (mean VHG: {vhg_df["VHG"].mean():.6f}, n={len(vhg_df)})', 
                    fontsize=12)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Add text with flow direction
        mean_vhg = vhg_df['VHG'].mean()
        if mean_vhg > 0.01:
            flow_text = "Predominantly DOWNWARD (recharge)"
            color = 'blue'
        elif mean_vhg < -0.01:
            flow_text = "Predominantly UPWARD (discharge)"
            color = 'orange'
        else:
            flow_text = "Minimal vertical flow"
            color = 'gray'
        
        ax.text(0.02, 0.98, flow_text, transform=ax.transAxes,
               fontsize=10, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    
    # Set x-label on bottom plot only
    axes[-1].set_xlabel('Date', fontsize=12)
    
    plt.tight_layout()
    
    # Save
    plot_file = output_dir / 'VHG_timeseries_all_sites.png'
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"VHG plot saved to: {plot_file.name}")
    plt.close()

    # Create summary box plot
    create_vhg_boxplot(vhg_results, output_dir)

In [35]:
if __name__ == "__main__":
    # Set your processed data directory
    data_dir = Path(r"C:\Users\leila\Dropbox\MayoWetlands\LevelloggerData\2025\processed")
    
    # Calculate VHG for all paired sites
    vhg_results = calculate_all_vhg(data_dir)
    
    # Create visualizations
    if vhg_results:
        output_dir = data_dir / 'VHG_analysis'
        plot_vhg_all_sites(vhg_results, output_dir)
        
        # NEW: Calculate August 1 - September 1 statistics
        august_stats = calculate_vhg_august_stats(vhg_results, output_dir)  # <-- ADDED THIS LINE
        
        print("\n" + "=" * 80)
        print("VHG ANALYSIS COMPLETE!")
        print("=" * 80)
        print(f"Results saved to: {output_dir}")
        print("\nFiles created:")
        print("  - Individual site VHG: [SITE]_VHG.csv")
        print("  - Paired water levels CSV: [SITE]_paired_water_levels.csv")
        print("  - Paired water levels plot: [SITE]_paired_water_levels.png")
        print("  - Combined VHG: all_sites_VHG.csv")
        print("  - Summary statistics: VHG_summary_statistics.csv")
        print("  - August statistics: VHG_August_stats.csv")
        print("  - Time series plot: VHG_timeseries_all_sites.png")
        print("  - Box plot comparison: VHG_boxplot_comparison.png")

VERTICAL HYDRAULIC GRADIENT (VHG) CALCULATION
Data directory: C:\Users\leila\Dropbox\MayoWetlands\LevelloggerData\2025\processed
Output directory: C:\Users\leila\Dropbox\MayoWetlands\LevelloggerData\2025\processed\VHG_analysis

Identifying sites with paired well-piezometer data...

Found 10 sites with paired data:
  - APA
  - CCA
  - ELA
  - ELB
  - GCB
  - HLA
  - MCB
  - MLW
  - STB
  - STS


Processing site: APA
------------------------------------------------------------
Well logger:  APA_well
  Sensor depth: 75.9 cm
  Records: 9619
  Date range: 2025-06-11 14:00:00 to 2025-09-19 18:30:00
Piezo logger: APA_piezo
  Sensor depth: 241.1 cm
  Records: 9622
  Date range: 2025-06-11 14:00:00 to 2025-09-19 19:15:00
Matched records: 9619
Vertical separation: 165.2 cm

VHG Statistics:
  Mean: -0.112857
  Median: -0.107748
  Std: 0.053738
  Min: -1.396489
  Max: 0.286320
  Interpretation: UPWARD flow (discharge conditions)
  Downward flow: 0.0% of time
  Upward flow: 100.0% of time
  Saved t

C:\Users\leila\AppData\Local\Temp\ipykernel_1316\2524202566.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)


VHG boxplot saved to: VHG_boxplot_comparison.png

CALCULATING VHG STATISTICS FOR AUGUST 1 - SEPTEMBER 1

Processing APA...
  Found 3072 records
  Date range: 2025-08-01 00:00:00 to 2025-09-01 23:45:00
  VHG average: -0.107919
  VHG median:  -0.108354
  VHG max:     -0.078692
  VHG min:     -0.142252
  VHG stdev:   0.003567
  VHG range:   0.063559

Processing CCA...
  Found 3072 records
  Date range: 2025-08-01 00:00:00 to 2025-09-01 23:45:00
  VHG average: -0.029695
  VHG median:  -0.029880
  VHG max:     -0.012616
  VHG min:     -0.047809
  VHG stdev:   0.008172
  VHG range:   0.035193

Processing ELA...
  Found 3072 records
  Date range: 2025-08-01 00:00:00 to 2025-09-01 23:45:00
  VHG average: -0.880016
  VHG median:  -0.852273
  VHG max:     -0.156250
  VHG min:     -1.721591
  VHG stdev:   0.455304
  VHG range:   1.565341

Processing ELB...
  Found 3072 records
  Date range: 2025-08-01 00:00:00 to 2025-09-01 23:45:00
  VHG average: -0.832203
  VHG median:  -0.814565
  VHG max:    